In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
def E_and_I_network(t, y, tau_I):
    M_EE = 1.25
    M_EI = -1.0
    M_IE = 1.0
    M_II = 0

    tau_E = 10e-3 #s
    g_I = 10 #Hz
    g_E = -10 #Hz

    rE, rI = y 

    rhs_E = (M_EE * rE + M_EI * rI - g_E) / tau_E
    rhs_I = (M_IE * rE + M_II * rI - g_I) / tau_I

    if rhs_E < 0:
        rhs_E = 0
    if rhs_I < 0:
        rhs_I = 0
    
    drE_dt = - rE/tau_E + rhs_E
    drI_dt = - rI/tau_I + rhs_I

    return np.array([drE_dt, drI_dt])


tau_I_values = np.array([30e-3, 50e-3])#np.arange(20e-3, 60e-3, 10e-3)
t_span = (0, 3)

y0 = np.array([0., 0.])

for tau_I in tau_I_values:
    sol = solve_ivp(E_and_I_network, t_span, y0, args=(tau_I,), t_eval=np.linspace(0, 1.5, 1000))
    plt.plot(sol.t, sol.y[0], label=f"$\\tau_I$={tau_I*1000:.0f} ms")


plt.title("rE(t) for different inhibitory time constants")
plt.xlabel("Time (s)")
plt.ylabel("Firing Rate (Hz)")
plt.legend()
plt.show()


In [ ]:


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for tau_I in tau_I_values:
    t = solutions[tau_I]["t"]
    rE = solutions[tau_I]["rE"]
    rI = solutions[tau_I]["rI"]
    
    label_text = f"$\\tau_I$={tau_I*1000:.0f} ms"
    
    ax1.plot(rE, rI, label=label_text)
    
    ax2.plot(t, rE, label=f"$r_E$ ({label_text})")
    ax2.plot(t, rI, '--', label=f"$r_I$ ({label_text})") 

ax1.set_xlabel(r"$r_E$ (Hz)")
ax1.set_ylabel(r"$r_I$ (Hz)")
ax1.set_title("Phase Plane")
ax1.legend()
ax1.grid(True)

ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Firing Rate (Hz)")
ax2.set_title("Firing Rates over Time")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
tau_scan = np.linspace(5e-3, 80e-3, 50)

amplitudes = []

for tau_I in tau_scan:

    sol = solve_ivp(
        E_and_I_network,
        (0, 3),
        y0,
        args=(tau_I,),
        t_eval=np.linspace(0, 3, 3000)
    )

    rE = sol.y[0]

    # discard transient
    steady = rE[2000:]

    amp = np.max(steady) - np.min(steady)

    amplitudes.append(amp)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(tau_scan * 1000, amplitudes)
plt.axvline(40, color='red', linestyle='--', label="Bifurcation point")
plt.grid(True)
plt.xlabel(r"$\tau_I$ (ms)")
plt.ylabel("Oscillation amplitude")
plt.title("Bifurcation diagram")


eig_real = []
M_EE = 1.25
M_EI = -1.0
M_IE = 1.0
M_II = 0

tau_E = 10e-3 #s
for tau_I in tau_scan:

    J = np.array([
        [(-1 + M_EE)/tau_E, M_EI/tau_E],
        [M_IE/tau_I, (-1 + M_II)/tau_I]
    ])

    eigvals = np.linalg.eigvals(J)

    eig_real.append(np.max(np.real(eigvals)))

plt.subplot(1, 2, 2)
plt.plot(tau_scan * 1000, eig_real)

plt.axhline(0, color='red', linestyle='--')
plt.grid(True)
plt.xlabel(r"$\tau_I$ (ms)")
plt.ylabel("Largest Re(eigenvalue)")
plt.title("Stability boundary")
plt.show()

del M_EE, M_EI, M_IE, M_II, tau_E

Sigmoid activation

In [ ]:
def activation_sigmoid(x, slope=0.05, x0=15.0):
    return 100 / (1 + np.exp(-slope * (x - x0)))

def E_and_I_network_sigmoid(t, y, tau_I, slope=0.05, x0=15.0):
    M_EE = 1.25
    M_EI = -1.0
    M_IE = 1.0
    M_II = 0

    tau_E = 10e-3 #s
    g_I = 10 #Hz
    g_E = -10 #Hz

    rE, rI = y 

    input_E = M_EE * rE + M_EI * rI - g_E
    input_I = M_IE * rE + M_II * rI - g_I

    rhs_E = -(rE/tau_E) + activation_sigmoid(input_E, slope, x0) / tau_E
    rhs_I = -(rI/tau_I) + activation_sigmoid(input_I, slope, x0) / tau_I

    drE_dt = rhs_E
    drI_dt = rhs_I

    return np.array([drE_dt, drI_dt])

tau_I_values = np.arange(10e-3, 60e-3, 10e-3)
t_span = (0, 3)

y0 = np.array([0., 0.])
beta = 0.15
x0 = 15.0
for tau_I in tau_I_values:
    sol = solve_ivp(E_and_I_network_sigmoid, t_span, y0, args=(tau_I, beta, x0), t_eval=np.linspace(0, 1.5, 1000))
    plt.plot(sol.t, sol.y[0], label=f"$\\tau_I$={tau_I*1000:.0f} ms")

plt.title("rE(t) for different inhibitory time constants (Sigmoid Activation)")
plt.xlabel("Time (s)")
plt.ylabel("Firing Rate (Hz)")
plt.legend()
plt.show()

solutions = {}

for tau_I in tau_I_values:

    sol = solve_ivp(
        E_and_I_network_sigmoid,
        t_span,
        y0,
        args=(tau_I, beta, x0),
        t_eval=np.linspace(0, 1.5, 1000)
    )

    solutions[tau_I] = {
        "t": sol.t,
        "rE": sol.y[0],
        "rI": sol.y[1]
    }

plt.figure(figsize=(6,6))

for tau_I in tau_I_values:

    rE = solutions[tau_I]["rE"]
    rI = solutions[tau_I]["rI"]

    plt.plot(rE, rI,
             label=f"$\\tau_I$={tau_I*1000:.0f} ms")

plt.xlabel(r"$r_E$ (Hz)")
plt.ylabel(r"$r_I$ (Hz)")
plt.title("Phase Plane")
plt.legend()
plt.show()

Nullclines

In [ ]:
rE_vals = np.linspace(0, 100, 1000)
rI_vals = np.linspace(0, 100, 1000)
R_E, R_I = np.meshgrid(rE_vals, rI_vals)

#Constants and parameters
M_EE = 1.25
M_EI = -1.0
M_IE = 1.0
M_II = 0

tau_E = 10e-3 #s
g_I = 10 #Hz
g_E = -10 #Hz

tau_I = 30e-3 #s
tau_E = 10e-3 #s

beta = 0.05
x0 = 15.0

input_E = M_EE * R_E + M_EI * R_I - g_E
input_I = M_IE * R_E + M_II * R_I - g_I
dRE = (-R_E + activation_sigmoid(input_E, beta, x0)) / tau_E
dRI = (-R_I + activation_sigmoid(input_I, beta, x0)) / tau_I

from scipy.optimize import fsolve

def fixed_point(vars, tau_I, beta, x0):
    rE, rI = vars

    input_E = M_EE*rE + M_EI*rI - g_E
    input_I = M_IE*rE + M_II*rI - g_I

    f1 = (-rE + activation_sigmoid(input_E, beta, x0))/tau_E
    f2 = (-rI + activation_sigmoid(input_I, beta, x0))/tau_I

    return [f1, f2]

fp = fsolve(fixed_point, [20,20], args=(tau_I, beta, x0))

print(fp)

def sigmoid_derivative(x, beta=0.2, x0=15):

    s = 1/(1 + np.exp(-beta*(x-x0)))

    return 100*beta*s*(1-s)

rE_fp, rI_fp = fp

uE = M_EE*rE_fp + M_EI*rI_fp - g_E
uI = M_IE*rE_fp + M_II*rI_fp - g_I

FEp = sigmoid_derivative(uE, beta, x0)
FIp = sigmoid_derivative(uI, beta, x0)

J = np.array([
    [(-1 + FEp*M_EE)/tau_E, (FEp*M_EI)/tau_E],
    [(FIp*M_IE)/tau_I, (-1 + FIp*M_II)/tau_I]
])

eigvals = np.linalg.eigvals(J)

print(eigvals)


# E-nullcline
plt.contour(R_E, R_I, dRE, levels=[0], colors='blue', )

# I-nullcline
plt.contour(R_E, R_I, dRI, levels=[0], colors='red')

plt.xlabel(r"$r_E$")
plt.ylabel(r"$r_I$")
plt.scatter(fp[0], fp[1], color='black', s=100)

plt.title("Nullclines of E-I Network with Sigmoid Activation")

plt.show()




Two parameter variation study - ($\tau_{I}$, $\beta$)

In [ ]:
tau_I_vals = np.linspace(10e-3, 80e-3, 10)
beta_vals = np.linspace(0.01, 0.3, 30)

#Constants and parameters
M_EE = 1.25
M_EI = -1.0
M_IE = 1.0
M_II = 0

tau_E = 10e-3 #s
g_I = 10 #Hz
g_E = -10 #Hz

x0 = 15.0

results = {}

initial_conditions = []

rE_init_vals = np.linspace(0, 100, 10)
rI_init_vals = np.linspace(0, 100, 10)

for rE0 in rE_init_vals:
    for rI0 in rI_init_vals:
        initial_conditions.append([rE0, rI0])

final_states = []

for y0 in initial_conditions:

    sol = solve_ivp(
        E_and_I_network_sigmoid,
        (0, 5),
        y0,
        args=(tau_I, beta, x0),
        t_eval=np.linspace(0, 5, 5000)
    )

    rE_final = sol.y[0][-1]
    rI_final = sol.y[1][-1]

    final_states.append([rE_final, rI_final])

final_states = np.array(final_states)

plt.figure(figsize=(7,7))

plt.scatter(
    final_states[:,0],
    final_states[:,1],
    s=30
)

plt.xlabel(r"$r_E$")
plt.ylabel(r"$r_I$")
plt.title("Final states after dynamical relaxation")

plt.show()



In [ ]:
tau_I_values = np.linspace(10e-3, 60e-3, 6)
beta_values = np.linspace(0.02, 0.3, 6)

results = {}

for tau_I in tau_I_values:

    results[tau_I] = {}

    for beta in beta_values:
        initial_conditions = []

        rE_init_vals = np.linspace(0, 100, 5)
        rI_init_vals = np.linspace(0, 100, 5)

        for rE0 in rE_init_vals:
            for rI0 in rI_init_vals:
                initial_conditions.append([rE0, rI0])
        final_states = []

        for y0 in initial_conditions:

            sol = solve_ivp(
                E_and_I_network_sigmoid,
                (0,5),
                y0,
                args=(tau_I, beta, x0),
                t_eval=np.linspace(0,5,5000)
            )

            rE_final = sol.y[0][-1]
            rI_final = sol.y[1][-1]

            final_states.append([rE_final, rI_final])

        final_states = np.array(final_states)
        candidate_points = []

        tol = 1e-1

        for point in final_states:

            unique = True

            for fp in candidate_points:

                if np.linalg.norm(point-fp) < tol:
                    unique = False
                    break

            if unique:
                candidate_points.append(point)

        candidate_points = np.array(candidate_points)

        results[tau_I][beta] = {
            "candidate_points": candidate_points,
            "final_states": final_states
        }

num_candidates = np.zeros((len(tau_I_values), len(beta_values)))

for i, tau_I in enumerate(tau_I_values):

    for j, beta in enumerate(beta_values):

        cps = results[tau_I][beta]["candidate_points"]

        num_candidates[i,j] = len(cps)

plt.figure(figsize=(7,5))

plt.imshow(
    num_candidates,
    origin='lower',
    aspect='auto',
    extent=[
        beta_values[0],
        beta_values[-1],
        tau_I_values[0]*1000,
        tau_I_values[-1]*1000
    ]
)

plt.colorbar(label='Number of candidate attractors')

plt.xlabel(r'$\beta$')
plt.ylabel(r'$\tau_I$ (ms)')
plt.title('Attractor Structure')

plt.show()

In [ ]:
plt.figure(figsize=(7,7))

for tau_I in tau_I_values:

    for beta in beta_values:

        cps = results[tau_I][beta]["candidate_points"]

        plt.scatter(
            cps[:,0],
            cps[:,1],
            label=f'b={beta:.2f}'
        )

plt.xlabel(r'$r_E$')
plt.ylabel(r'$r_I$')

plt.title('Candidate Attractor Locations')

plt.show()

In [ ]:
tau_I = 30e-3
beta = 0.076

# trajectory
sol = solve_ivp(
    E_and_I_network_sigmoid,
    (0,5),
    [20,20],
    args=(tau_I, beta, x0),
    t_eval=np.linspace(0,5,5000)
)

# grid
rE_vals = np.linspace(0,100,30)
rI_vals = np.linspace(0,100,30)

RE, RI = np.meshgrid(rE_vals, rI_vals)

input_E = M_EE*RE + M_EI*RI - g_E
input_I = M_IE*RE + M_II*RI - g_I

dRE = (-RE + activation_sigmoid(input_E,beta,x0))/tau_E
dRI = (-RI + activation_sigmoid(input_I,beta,x0))/tau_I

plt.figure(figsize=(8,8))

# vector field
plt.quiver(
    RE, RI,
    dRE, dRI,
    color='gray',
    alpha=0.5
)

# nullclines
plt.contour(RE, RI, dRE, levels=[0], colors='blue')
plt.contour(RE, RI, dRI, levels=[0], colors='red')

# trajectory
plt.plot(sol.y[0], sol.y[1],
         color='black',
         linewidth=2)

plt.xlabel(r"$r_E$")
plt.ylabel(r"$r_I$")

plt.title("Phase Portrait")

plt.show()

In [ ]:
fig, axes = plt.subplots(
    len(tau_I_values),
    len(beta_values),
    figsize=(18,18),
    sharex=True,
    sharey=True
)

for i, tau_I in enumerate(tau_I_values):

    for j, beta in enumerate(beta_values):

        ax = axes[i,j]

        cps = results[tau_I][beta]["candidate_points"]

        ax.scatter(
            cps[:,0],
            cps[:,1],
            s=30,
            color='black'
        )

        ax.set_title(
            rf"$\tau_I$={tau_I*1000:.0f}ms"
            "\n"
            rf"$\beta$={beta:.2f}",
            fontsize=9
        )

        ax.set_xlim(0,100)
        ax.set_ylim(0,100)

        # optional: cleaner look
        ax.tick_params(labelsize=6)

plt.tight_layout()

fig.supxlabel(r"$r_E$")
fig.supylabel(r"$r_I$")

plt.show()